In [ ]:
from apeGmsh import apeGmsh
from apeGmsh import VTKExport, write_vtu, write_vtu_series, VTK_QUAD
import numpy as np
import openseespy.opensees as ops
import gmsh

# Shell I-Beam Modal Analysis — ParaView Visualization

**Same IPE200 cantilever modal analysis**, but instead of Gmsh views we
export the mesh and results to **VTU files** for visualization in
[ParaView](https://www.paraview.org/).

**Why ParaView?**

| Feature | Gmsh Views | ParaView (VTU) |
|---------|-----------|---------------|
| Built-in to workflow | ✓ | needs export |
| Scripting (Python) | limited | full (pvpython) |
| Animation / video | basic | publication-quality |
| Multi-field overlay | one view at a time | side-by-side, linked |
| Large models (>1M elems) | can lag | optimized |
| Filters (clip, slice, warp) | manual | built-in pipeline |
| Collaboration | .msh files | .vtu / .pvd (standard) |

**Export approach:**

pyGmsh includes a zero-dependency `VTKExport` module that writes VTK
UnstructuredGrid files (.vtu) using only numpy + stdlib XML. It produces:

- **Single .vtu file** — one snapshot with all scalar/vector fields
- **Time-series .pvd** — collection of .vtu files, one per mode shape,
  that ParaView treats as animation frames (step through modes)

In [ ]:
# ============================================================
# IPE200 cross-section & material
# ============================================================
h   = 200.0;  b  = 100.0;  tw = 5.6;  tf = 8.5
L   = 3000.0
d   = h - tf     # 191.5 mm

E   = 210000.0   # [MPa]
nu  = 0.3
rho = 7.85e-9    # [tonne/mm³]

# Mesh divisions
n_half_flange = 4
n_web_height  = 12
n_length      = 60

# Number of modes to compute
num_modes = 10

print(f"IPE200 cantilever: L = {L:.0f} mm, {num_modes} modes")

## Part 1 — Geometry, Mesh & Modal Analysis

Same 5-surface mid-surface I-beam and transfinite quad mesh as the
previous examples. Condensed here for brevity — see
`example_ibeam_modal.ipynb` for the fully commented version.

In [ ]:
g = apeGmsh(model_name="IPE200_ParaView", verbose=True)
g.begin()

# --- 12 corner points ---
p1  = g.model.geometry.add_point(0, -b/2, 0,  lc=30)
p2  = g.model.geometry.add_point(0,  0,   0,  lc=30)
p3  = g.model.geometry.add_point(0,  b/2, 0,  lc=30)
p4  = g.model.geometry.add_point(0,  0,   d,  lc=30)
p5  = g.model.geometry.add_point(0, -b/2, d,  lc=30)
p6  = g.model.geometry.add_point(0,  b/2, d,  lc=30)
p7  = g.model.geometry.add_point(L, -b/2, 0,  lc=30)
p8  = g.model.geometry.add_point(L,  0,   0,  lc=30)
p9  = g.model.geometry.add_point(L,  b/2, 0,  lc=30)
p10 = g.model.geometry.add_point(L,  0,   d,  lc=30)
p11 = g.model.geometry.add_point(L, -b/2, d,  lc=30)
p12 = g.model.geometry.add_point(L,  b/2, d,  lc=30)

# --- Lines ---
c1 = g.model.geometry.add_line(p1, p2);  c2 = g.model.geometry.add_line(p2, p3)
c3 = g.model.geometry.add_line(p2, p4);  c4 = g.model.geometry.add_line(p5, p4)
c5 = g.model.geometry.add_line(p4, p6)
c6  = g.model.geometry.add_line(p7,  p8);   c7  = g.model.geometry.add_line(p8,  p9)
c8  = g.model.geometry.add_line(p8,  p10);  c9  = g.model.geometry.add_line(p11, p10)
c10 = g.model.geometry.add_line(p10, p12)
e1 = g.model.geometry.add_line(p1, p7);   e2 = g.model.geometry.add_line(p2, p8)
e3 = g.model.geometry.add_line(p3, p9);   e4 = g.model.geometry.add_line(p4, p10)
e5 = g.model.geometry.add_line(p5, p11);  e6 = g.model.geometry.add_line(p6, p12)

# --- 5 Surfaces ---
s_bf_l = g.model.geometry.add_plane_surface(g.model.geometry.add_curve_loop([c1, e2, -c6, -e1]))
s_bf_r = g.model.geometry.add_plane_surface(g.model.geometry.add_curve_loop([c2, e3, -c7, -e2]))
s_web  = g.model.geometry.add_plane_surface(g.model.geometry.add_curve_loop([c3, e4, -c8, -e2]))
s_tf_l = g.model.geometry.add_plane_surface(g.model.geometry.add_curve_loop([-c4, e5, c9, -e4]))
s_tf_r = g.model.geometry.add_plane_surface(g.model.geometry.add_curve_loop([c5, e6, -c10, -e4]))

# --- Physical groups ---
pg_flanges = g.physical.add(2, [s_bf_l, s_bf_r, s_tf_l, s_tf_r], name="Flanges")
pg_web     = g.physical.add(2, [s_web], name="Web")
pg_fixed   = g.physical.add(1, [c1, c2, c3, c4, c5], name="FixedEnd")

# --- Transfinite quad mesh ---
for c in [c1, c2, c4, c5, c6, c7, c9, c10]:
    gmsh.model.mesh.setTransfiniteCurve(c, n_half_flange + 1)
for c in [c3, c8]:
    gmsh.model.mesh.setTransfiniteCurve(c, n_web_height + 1)
for c in [e1, e2, e3, e4, e5, e6]:
    gmsh.model.mesh.setTransfiniteCurve(c, n_length + 1)
for s in [s_bf_l, s_bf_r, s_web, s_tf_l, s_tf_r]:
    gmsh.model.mesh.setTransfiniteSurface(s)
    gmsh.model.mesh.setRecombine(2, s)

g.mesh.generation.set_order(1)
g.mesh.generation.generate(2)
print(f"Mesh: {sum(len(t) for t in g.mesh.queries.get_elements(dim=2)['tags'])} quad elements")

In [ ]:
# --- Extract mesh data ---
fem = g.mesh.queries.get_fem_data(dim=2)
node_tags    = fem.node_ids
node_coords  = fem.node_coords
tag_to_idx   = {int(t): i for i, t in enumerate(fem.node_ids)}
connectivity = fem.connectivity
elem_tags    = fem.element_ids
used_tags    = set(fem.node_ids.tolist())

# Region classification
flange_elem_set = set()
for surf in [s_bf_l, s_bf_r, s_tf_l, s_tf_r]:
    for etags in g.mesh.queries.get_elements(dim=2, tag=surf)['tags']:
        flange_elem_set.update(etags.astype(int).tolist())
web_elem_set = set()
for etags in g.mesh.queries.get_elements(dim=2, tag=s_web)['tags']:
    web_elem_set.update(etags.astype(int).tolist())

fixed_nodes = g.physical.get_nodes(1, pg_fixed)['tags']

# --- Build OpenSees model ---
ops.wipe()
ops.model('basic', '-ndm', 3, '-ndf', 6)

gmsh_to_ops = {}
new_id = 0
for gtag, coords in zip(node_tags.astype(int), node_coords):
    if int(gtag) not in used_tags:
        continue
    new_id += 1
    gmsh_to_ops[int(gtag)] = new_id
    ops.node(new_id, float(coords[0]), float(coords[1]), float(coords[2]))

ops.section('ElasticMembranePlateSection', 1, E, nu, tf, rho)
ops.section('ElasticMembranePlateSection', 2, E, nu, tw, rho)

for i, (etag, row) in enumerate(zip(elem_tags, connectivity)):
    eid = i + 1
    nodes = [gmsh_to_ops[int(n)] for n in row]
    sec = 1 if etag in flange_elem_set else 2
    ops.element('ShellMITC4', eid, *nodes, sec)

for gtag in fixed_nodes.astype(int):
    if int(gtag) in gmsh_to_ops:
        ops.fix(gmsh_to_ops[int(gtag)], 1, 1, 1, 1, 1, 1)

# --- Eigenvalue analysis ---
eigenvalues = ops.eigen(num_modes)
frequencies = [np.sqrt(ev)/(2*np.pi) for ev in eigenvalues]

print(f"Model: {new_id} nodes, {connectivity.shape[0]} elements")
print(f"\nNatural frequencies:")
for i, f in enumerate(frequencies):
    print(f"  Mode {i+1}: {f:.3f} Hz")

In [ ]:
# --- Extract all mode shapes ---
nNode = len(node_tags)
mode_shapes = []

for m in range(1, num_modes + 1):
    phi = np.zeros((nNode, 6))
    for gtag, ops_id in gmsh_to_ops.items():
        idx = tag_to_idx[gtag]
        for dof in range(6):
            phi[idx, dof] = ops.nodeEigenvector(ops_id, m, dof + 1)
    mode_shapes.append(phi)

ops.wipe()
print(f"Extracted {num_modes} mode shapes ({nNode} nodes × 6 DOFs)")

## Part 2 — Export to VTU for ParaView

We use pyGmsh's `VTKExport` module to write `.vtu` files. Two export
strategies:

### Strategy A: Single .vtu with all modes as separate fields

Each mode shape becomes a named vector field ("Mode_1", "Mode_2", …)
in one file. In ParaView, you pick which field to display via the
dropdown and use the **Warp By Vector** filter to animate.

### Strategy B: Time-series .pvd (one .vtu per mode)

Each mode is a separate .vtu file, collected by a `.pvd` master file.
ParaView treats each mode as a "time step", so you can use the
**animation toolbar** to step through modes. The mode's frequency is
stored as the "time" value — visible in the status bar.

In [ ]:
# --- Strategy A: single .vtu file with all fields ---
#
# VTKExport wraps get_fem_data() and provides add_node_scalar/vector
# methods that mirror the Gmsh View API.

vtk = VTKExport(g, dim=2)

# Add each mode as a named vector field
for i in range(num_modes):
    phi3 = mode_shapes[i][:, :3]   # translational DOFs only (ux, uy, uz)
    mag  = np.sqrt(np.sum(phi3**2, axis=1))
    
    vtk.add_node_vector(f"Mode_{i+1}", phi3)
    vtk.add_node_scalar(f"Mode_{i+1}_mag", mag)

# Add a "region" field so ParaView can color by flange vs web
region = np.zeros(connectivity.shape[0])
for i, etag in enumerate(elem_tags):
    region[i] = 1.0 if etag in flange_elem_set else 2.0
vtk.add_elem_scalar("Region", region)

out_single = vtk.write("IPE200_modes_all.vtu")
print(f"Written: {out_single}")
print(f"  → Open in ParaView, select 'Mode_1' from the field dropdown")
print(f"  → Apply 'Warp By Vector' filter to see deformed shape")

In [ ]:
# --- Strategy B: .pvd time series (one .vtu per mode) ---
#
# This is the preferred approach for stepping through modes with
# ParaView's animation controls. Each mode gets its own .vtu file,
# and the .pvd collection file tells ParaView the ordering and
# "time" value (we use frequency as the time axis).

vtk_series = VTKExport(g, dim=2)

files = vtk_series.write_mode_series(
    "IPE200_modes",
    mode_shapes=[ms[:, :3] for ms in mode_shapes],
    frequencies=frequencies,
)

print(f"Written {len(files)} files:")
for f in files:
    print(f"  {f}")
print()
print("Open 'IPE200_modes.pvd' in ParaView:")
print("  1. Apply 'Warp By Vector' filter on 'ModeShape'")
print("  2. Set Scale Factor (~50-200 for visibility)")
print("  3. Use the animation toolbar ▶ to step through modes")
print("  4. The 'Time' value shown = frequency [Hz]")

## Part 3 — ParaView Visualization Guide

### Opening the files

1. **File → Open** → select `IPE200_modes.pvd` (the collection file)
2. Click **Apply** in the Properties panel

### Viewing mode shapes (Warp By Vector)

1. **Filters → Alphabetical → Warp By Vector**
2. Set **Vectors** = `ModeShape`
3. Set **Scale Factor** = `100` (adjust for visibility; the eigenvectors
   are normalized, so you need a large scale)
4. Click **Apply**
5. Color by `Magnitude` for a displacement-magnitude contour

### Stepping through modes

- The **animation toolbar** (▶, ⏮, ⏭) steps through modes
- Each mode's frequency appears as the "Time" value in the bottom status bar
- Or use the **Time Inspector** (View → Time Inspector) for a list

### Side-by-side comparison

1. **View → Split Horizontal** (or Vertical)
2. In each panel, load the same PVD file
3. Apply Warp By Vector in each
4. Set different mode numbers via the animation slider

### Making publication figures

1. Apply Warp By Vector + color by Magnitude
2. **View → Color Map Editor** → choose a perceptual colormap (viridis, coolwarm)
3. Add a **color bar**: right-click the pipeline entry → Toggle Color Legend
4. **File → Save Screenshot** → set resolution (e.g. 3000 × 2000)
5. For animations: **File → Save Animation** → choose AVI or PNG sequence

### Useful filters for structural analysis

| Filter | What it does |
|--------|-------------|
| **Warp By Vector** | Deforms the mesh by a vector field |
| **Clip** | Cuts the model with a plane (see internal stress) |
| **Slice** | Extracts a cross-section plane |
| **Contour** | Iso-surfaces of a scalar field |
| **Calculator** | Derive new fields (e.g., von Mises from components) |
| **Plot Over Line** | Extracts values along a line → XY plot |
| **Threshold** | Show only elements where a field is in a range |

In [ ]:
# --- Quick verification: read back a VTU file and check structure ---
import xml.etree.ElementTree as ET

tree = ET.parse("IPE200_modes_all.vtu")
root = tree.getroot()
piece = root.find(".//Piece")
n_pts  = piece.get("NumberOfPoints")
n_cells = piece.get("NumberOfCells")

pd_names = [da.get("Name") for da in piece.findall(".//PointData/DataArray")]
cd_names = [da.get("Name") for da in piece.findall(".//CellData/DataArray")]

print(f"VTU file verification:")
print(f"  Points: {n_pts},  Cells: {n_cells}")
print(f"  Point fields ({len(pd_names)}): {', '.join(pd_names[:6])}{'...' if len(pd_names)>6 else ''}")
print(f"  Cell fields  ({len(cd_names)}): {', '.join(cd_names)}")

# Verify PVD
tree_pvd = ET.parse("IPE200_modes.pvd")
datasets = tree_pvd.findall(".//DataSet")
print(f"\nPVD collection: {len(datasets)} time steps")
for ds in datasets[:3]:
    print(f"  t = {float(ds.get('timestep')):.3f} Hz → {ds.get('file')}")
if len(datasets) > 3:
    print(f"  ... ({len(datasets)-3} more)")

## Part 4 — Gmsh Views (for comparison)

We still create Gmsh views so you can compare both visualization
approaches side by side.

In [ ]:
# --- Gmsh views (same as the modal example) ---
node_tag_list = list(node_tags.astype(int))

for i in range(min(num_modes, 6)):
    phi = mode_shapes[i]
    g.view.add_node_vector(
        f"Mode {i+1} — f={frequencies[i]:.2f} Hz",
        node_tag_list, phi[:, :3])

mag1 = np.sqrt(np.sum(mode_shapes[0][:, :3]**2, axis=1))
g.view.add_node_scalar("Mode 1 |u|", node_tag_list, mag1)

print(f"Created {g.view.count()} Gmsh views")

In [ ]:
# --- Summary: mode classification ---
print(f"{'Mode':>4s}  {'f [Hz]':>10s}  {'Dominant':>10s}  {'Type':>20s}")
print("-" * 50)

for i in range(num_modes):
    phi = mode_shapes[i]
    rms = [np.sqrt(np.mean(phi[:, j]**2)) for j in range(6)]
    max_dof = np.argmax(rms[:3])
    dof_labels = ['ux', 'uy', 'uz', 'rx', 'ry', 'rz']
    
    if max_dof == 1:   mtype = "Weak-axis bending"
    elif max_dof == 2: mtype = "Strong-axis bending"
    elif max_dof == 0: mtype = "Axial / coupled"
    else:              mtype = "Unknown"
    if rms[3] > 1.5 * max(rms[:3]): mtype = "Torsional"
    
    print(f"{i+1:4d}  {frequencies[i]:10.3f}  {dof_labels[np.argmax(rms)]:>10s}  {mtype:>20s}")

## Part 5 — Launch Gmsh GUI & Finalize

In [ ]:
g.launch_gui()

In [ ]:
g.end()

print("\nOutput files for ParaView:")
print("  IPE200_modes_all.vtu  — single file with all modes")
print("  IPE200_modes.pvd      — time series (open this one)")
print("  IPE200_modes_000.vtu … IPE200_modes_009.vtu — per-mode files")